# Week 8 Overview

This week will be a mix of data joining/merging problems and linear algebra. 
The first 5 problems are data cleaning and the final 4 problems are linear algebra. 

There are multiple ways to combine data. These methods are common cross multiple languages like pandas, SQL, and R. At times the naming is different but the general concepts apply. 

### **Joining or Merging**
This is a process of combining two datasets by adding the columns of one dataset to the other by some logical relationship between the columns. 

In SQL we call this joining but pandas has two functions:

**merge** - The default behavor for merge is to combine on columns matching.

**join** - The default behavor for join is to combine on the column index matching. 

Often times I will colloquially use the word "join" for either merging or joining in pandas. 

Left Dataset
| key    | value |
| -------- | ------- |
| A1  | $250    |
| A2 | $80     |
| A3    | $420    |

Right Dataset
| key    | different_value |
| -------- | ------- |
| A1  | cat    |
| A2 | dog     |
| A3    | apple    |

Data Joined on key

| key    | value | different_value |
| -------- | ------- | ------- |
| A1  | $250    | cat |
| A2 | $80     | dog |
| A3    | $420    | apple |

Typically we refer to the starting dataset as the left dataset and the one being added as the right. 

The logic is typically that there is the same value in a specific column in both datasets. SQL allows for slightly more advanced logic which we will learn next quarter. Today we will focus on just columns matching. 

There are different types of joins that you will explore in this notebook (inner, outer, left, right, cross). The typical visual that is used to illustration these concepts in Ven Diagrams. If you are getting stuck trying to pick the right join type search for "types of joins" and look at the pictures that come up.


## **Concat or Union**

This is a process of combining two dataset by adding the rows of one dataset to the end of another. There is no logic required for this. This is called conact in pandas and union in SQL. 

In most version of SQL you are required to have the same columns in both datasets. In pandas you don't have to. If I concatenate the two dataset above in pandas I would get:

| key    | value | different_value |
| -------- | ------- | ------- |
| A1  | $250    | null |
| A2 | $80     | null |
| A3    | $420    | null |
| A1  | null    | cat |
| A2 | null     | dog |
| A3    | null    | apple |

However if my right dataset looked like this:

| key    | value |
| -------- | ------- |
| A1  | cat    |
| A2 | dog     |
| A3    | apple    |

then I could union them in SQL or concat in pandas to get:

| key    | value |
| -------- | ------- |
| A1  | $250    |
| A2 | $80     |
| A3    | $420    |
| A1  | cat    |
| A2 | dog     |
| A3    | apple    |



In [1]:
import pandas as pd
import numpy as np

In [2]:
df_1 = pd.DataFrame({"ints": range(100)})
df_2 = pd.DataFrame({"ints": range(-10, 10)}, index=range(-10, 10))


df_1['threes'] = np.floor(df_1['ints']/3) * 3

df_2['evens'] = df_2['ints']*2
df_2['threes'] = np.floor(df_2['ints']/3) * 3

### Problem 1:

Your first task will be to create a dataset by `merging` `df_1` and `df_2` on the `ints` column where the match on both sides. The results will be a dataframe with 10 rows and 5 columns.

You will then create the same dataframe by using the `join` function and joining the two datasets where the indexes are equal. There will be a little more work of handle column duplication so look up the error and figure out arguments to set. How many columns do you get in this case?

In [33]:
merged_df = df_1.merge(
    df_2.reset_index(), 
    on="ints",
    how="inner"
)

print("Merged Shape:", merged_df.shape)
merged_df

Merged Shape: (10, 7)


,ints,threes_x,threes_string_x,index,evens,threes_y,threes_string_y
0,0,0.0,0.0,0,0,0.0,0.0
1,1,0.0,0.0,1,2,0.0,0.0
2,2,0.0,0.0,2,4,0.0,0.0
3,3,3.0,3.0,3,6,3.0,3.0
4,4,3.0,3.0,4,8,3.0,3.0
5,5,3.0,3.0,5,10,3.0,3.0
6,6,6.0,6.0,6,12,6.0,6.0
7,7,6.0,6.0,7,14,6.0,6.0
8,8,6.0,6.0,8,16,6.0,6.0
9,9,9.0,9.0,9,18,9.0,9.0


In [4]:
joined_df = df_1.join(
    df_2,
    how="inner",
    lsuffix="_df1",
    rsuffix="_df2"
)

print("\nJoined Shape:", joined_df.shape)
print(joined_df.head())


Joined Shape: (10, 5)
   ints_df1  threes_df1  ints_df2  evens  threes_df2
0         0         0.0         0      0         0.0
1         1         0.0         1      2         0.0
2         2         0.0         2      4         0.0
3         3         3.0         3      6         3.0
4         4         3.0         4      8         3.0


## Problem 2:

Next you will perform the same merge as above three times with the following modifications:

* You want to keep all rows in `df_1` even if there is no match found in `df_2`
* You want to keep all rows in `df_2` even if there is no match found in `df_1`
* You want to keep all rows in `df_1` and `df_2` even if there is no match found in the other dataframe


How many rows do you end up with in each case? 

Think through a scenario where you might want to do this and add it as a comment above each merge. 

In [14]:
# Only ints 0–9 appear in both dataframes.
# The rest (10–99) don’t exist in df_2, so those columns become NaN.
left_merge = df_1.merge(
    df_2.reset_index(),
    on="ints",
    how="left"
)

print("Left Join Shape:", left_merge.shape) 

Left Join Shape: (100, 6)


In [36]:
# Only ints 0–9 match between the dataframes.
# Values -10 to -1 aren’t in df_1, so those columns show NaN.
right_merge = df_1.merge(
    df_2.reset_index(),
    on="ints",
    how="right"
)

print("Right Join Shape:", right_merge.shape) 

Right Join Shape: (20, 7)


In [37]:
# This keeps every int from both dataframes (0–99 and -10–9).
# If a value doesn’t exist on one side, pandas fills it with NaN.
outer_merge = df_1.merge(
    df_2.reset_index(),
    on="ints",
    how="outer"
)

print("Outer Join Shape:", outer_merge.shape)

Outer Join Shape: (110, 7)


### Problem 3

Now we are going to merge on columns that are not the same. Merge on the following:

* Merge `df_1` and `df_2` where `df_1.ints = df_2.evens`, only keep rows where there is a value for either dataframe
* Merge `df_1` and `df_2` where `df_1.ints = df_2.threes`, only keep rows where there is a value for either dataframe
* Merge `df_1` and `df_2` where `df_1.ints = df_2.threes`, keep all rows from `df_1` even if there is no match found in `df_2`
* Merge `df_1` and `df_2` where `df_1.threes = df_2.threes`, only keep rows where there is a value for either dataframe


How many rows do you end up with in each case? Are there any duplications? (try: value_count)

Think through a scenario where you might want to do this and add it as a comment above each merge. 

In [20]:
# This matches df_1.ints to df_2.evens instead of matching the same column names.
# Since df_2.evens only contains even numbers from -20 to 18, only those values match.
m1 = df_1.merge(
    df_2,
    left_on="ints",
    right_on="evens",
    how="inner",
    suffixes=("_df1", "_df2")
)

print(m1["ints_df1"].value_counts().head())
print(m1["ints_df2"].value_counts().head())

ints_df1
0    1
2    1
4    1
6    1
8    1
Name: count, dtype: int64
ints_df2
0    1
1    1
2    1
3    1
4    1
Name: count, dtype: int64


In [22]:
# This matches df_1.ints to df_2.threes, so it’s grouping df_2 ints into buckets of 3.
# That can create repeats because multiple df_2 rows share the same threes value.
m2 = df_1.merge(
    df_2,
    left_on="ints",
    right_on="threes",
    how="inner",
    suffixes=("_df1", "_df2")
)

print("\n2) inner on df_1.ints = df_2.threes -> rows:", len(m2))
print("   duplicates in df_1 ints after merge (top):")
print(m2["ints_df1"].value_counts().head())


2) inner on df_1.ints = df_2.threes -> rows: 10
   duplicates in df_1 ints after merge (top):
ints_df1
0    3
3    3
6    3
9    1
Name: count, dtype: int64


In [26]:
# This is a left join, so we keep every row from df_1 even if df_2 doesn’t match.
# Some df_1 ints match the same df_2 threes bucket, so a few df_1 rows get duplicated.
m3 = df_1.merge(
    df_2,
    left_on="ints",
    right_on="threes",
    how="left",
    suffixes=("_df1", "_df2")
)

print("\n3) left on df_1.ints = df_2.threes -> rows:", len(m3))
print("   duplicates in df_1 ints after merge (top):")
print(m3["ints_df1"].value_counts().head())


3) left on df_1.ints = df_2.threes -> rows: 106
   duplicates in df_1 ints after merge (top):
ints_df1
0     3
3     3
6     3
63    1
73    1
Name: count, dtype: int64


In [ ]:
# This matches rows where both dataframes fall into the same multiple-of-3 bucket.
# Since many ints share the same "threes" value, this creates a lot of duplicates.
m4 = df_1.merge(
    df_2,
    on="threes",
    how="inner",
    suffixes=("_df1", "_df2")
)

print("\n4) inner on df_1.threes = df_2.threes -> rows:", len(m4))
print("   duplicates in threes after merge (top):")
print(m4["threes"].value_counts().head())


4) inner on df_1.threes = df_2.threes -> rows: 30
   duplicates in threes after merge (top):
threes
0.0    9
3.0    9
6.0    9
9.0    3
Name: count, dtype: int64


### Problem 4

Add a new the column to `df_2` called `threes_string` that is the `threes` column converted to a string. Attempt to merge `df_1` and `df_2` where `df_1.threes = df_2.threes_string` with an inner join. What happens? Why?

In [ ]:
df_2

# Convert df_2.threes to string
df_2["threes_string"] = df_2["threes"].astype(str)

# This merge would fail because df_1.threes is numeric and df_2.threes_string is text.
# Pandas requires both merge keys to be the same data type.

# Convert df_1.threes to string so the types match
# df_1["threes_string"] = df_1["threes"].astype(str)

mismatch_merge = df_1.merge(
    df_2,
    left_on="threes_string",
    right_on="threes_string",
    how="inner"
)

# print("Rows returned:", len(mismatch_merge))
# print(mismatch_merge.head())
mismatch_merge

,ints_x,threes_x,threes_string,ints_y,evens,threes_y
0,0,0.0,0.0,0,0,0.0
1,0,0.0,0.0,1,2,0.0
2,0,0.0,0.0,2,4,0.0
3,1,0.0,0.0,0,0,0.0
4,1,0.0,0.0,1,2,0.0
5,1,0.0,0.0,2,4,0.0
6,2,0.0,0.0,0,0,0.0
7,2,0.0,0.0,1,2,0.0
8,2,0.0,0.0,2,4,0.0
9,3,3.0,3.0,3,6,3.0


### Problem 5

Now you will play around with `pd.concat` by doing the following:

* Concatenate `df_1` and `df_2` keeping all rows, columns and indexes
* Concatenate `df_1` and `df_2` keeping all rows and columns but ignore the indexes from the orginal dataframes and instead have the index on this dataframe be zero to the number of rows.
* Concatenate `df_1` and `df_2` keeping all rows and indexes the same but only keeping columns that exist in both dataframes


In [29]:
# 1) Keep all rows, columns, and original indexes (default behavior)
# This just stacks df_1 and df_2 on top of each other and keeps their indexes.
concat_all = pd.concat([df_1, df_2])

print("1) Shape (keep indexes):", concat_all.shape)
print(concat_all.head())


# 2) Keep all rows and columns but reset the index
# ignore_index=True resets the index so it runs from 0 to total_rows - 1.
concat_reset_index = pd.concat([df_1, df_2], ignore_index=True)

print("\n2) Shape (reset index):", concat_reset_index.shape)
print(concat_reset_index.head())


# 3) Keep all rows and original indexes but only shared columns
# join='inner' keeps only the columns that both dataframes have in common.
concat_common_cols = pd.concat([df_1, df_2], join="inner")

print("\n3) Shape (common columns only):", concat_common_cols.shape)
print(concat_common_cols.head())

1) Shape (keep indexes): (120, 4)
   ints  threes threes_string  evens
0     0     0.0           0.0    NaN
1     1     0.0           0.0    NaN
2     2     0.0           0.0    NaN
3     3     3.0           3.0    NaN
4     4     3.0           3.0    NaN

2) Shape (reset index): (120, 4)
   ints  threes threes_string  evens
0     0     0.0           0.0    NaN
1     1     0.0           0.0    NaN
2     2     0.0           0.0    NaN
3     3     3.0           3.0    NaN
4     4     3.0           3.0    NaN

3) Shape (common columns only): (120, 3)
   ints  threes threes_string
0     0     0.0           0.0
1     1     0.0           0.0
2     2     0.0           0.0
3     3     3.0           3.0
4     4     3.0           3.0


## Linear Algebra: Rank and Column Space

### Problem 6
You will now learn how to create random matrices with arbitrary rank (subject to the constraints about matrix sizes, etc.). To create an $m \times n$ matrix with rank $r$, multiply a random $m \times r$ matrix with a random $r \times n$ matrix. Implement this in Python and confirm that the rank is indeed $r$. 

What happens if you set $r > min{M,N}$, and why does that happen?

In [ ]:
# Define dimensions and desired rank
m, n = 5, 8
r = 3

# Step 1: Create two random matrices that sandwich the rank r
# Matrix A is m x r, Matrix B is r x n
A = np.random.randn(m, r)
B = np.random.randn(r, n)

# Step 2: Multiply them to get an m x n matrix
C = A @ B

# Step 3: Check the rank of the resulting matrix
rank_C = np.linalg.matrix_rank(C)

print(f"Matrix shape: {C.shape}")
print(f"Target rank: {r}")
print(f"Actual rank: {rank_C}")

Matrix shape: (5, 8)
Target rank: 3
Actual rank: 3


### Problem 7
Interestingly, the matrices $A$, $A^T$, $A^T A$, and $AA^T$ all have the same rank. Write code to demonstrate this, using random matrices of various sizes, shapes (square, tall, wide), and ranks. Create a total of 6 random, two of each size that have different sizes and ranks. 

### Problem 8

Demonstrate the addition rule of matrix rank $(r(A + B) ≤ r(A) + r(B))$ by creating three pairs of rank-1 matrices that have a sum with 
1. rank-0
2. rank-1
3. rank-2

Then repeat this exercise using matrix multiplication instead of addition.

### Problem 9

The goal of this exercise is to answer the question is $v \in C(A)$?

Create a rank-3 matrix $A \in \mathbb{R}^{4 \times 3}$ and vector $v \in \mathbb{R}^{4}$ using numbers randomly drawn from a normal distribution. 

Follow the algorithm described in the [In the Column Space?](https://learning.oreilly.com/library/view/practical-linear-algebra/9781098120603/ch06.html#id335) section of Practical Linear Algebra to determine whether the vector is in the column space of the matrix. 

Rerun the code multiple times to see whether you find a consistent pattern. 

Next, use a $A \in \mathbb{R}^{4 \times 4}$ rank-4 matrix and a vector $v \in \mathbb{R}^{4}$ using numbers randomly drawn from a normal distribution. What happens in this case? Why?
